In [5]:
import xarray as xr
import pandas as pd
import numpy as np
from pandas.tseries.offsets import DateOffset

In [6]:
### CMA to make sure we use the same grid
ds_CMA = xr.open_dataset("/home/jupyter-vincent2/vincent/process_profiles/data/processed_2026/CMA_gridded.nc")

### ARGO
ds_ARGO = xr.open_dataset("/home/jupyter-vincent2/vincent/process_profiles/data/All/all_ARGO_2024.nc")

### CORA
ds_CORA = xr.open_dataset("/home/jupyter-vincent2/vincent/process_profiles/data/All/all_CORA_2024.nc")

ds_merge = xr.merge([ds_ARGO, ds_CORA], compat="override")
ds_merge = ds_merge.rename({"LATITUDE":"latitude", "LONGITUDE":"longitude"})
df_merge = ds_merge.mld.to_dataframe().reset_index()


def make_ds_cut(df, nb_bins=39, freq="M"):
    # detect coordinate column names
    lon_col = "longitude"
    lat_col = "latitude"
    time_col = "time"

    bins_dt = pd.date_range(
        start=df[time_col].min() + DateOffset(months=-1),
        end=df[time_col].max() + DateOffset(months=+1),
        freq=freq
    )

    cut_lat_label = pd.cut(df[lat_col], nb_bins)
    cut_lon_label = pd.cut(df[lon_col], nb_bins)
    cut_time_label = pd.cut(df[time_col], bins=bins_dt)

    df_cut_label = df.drop([lat_col, lon_col, time_col], axis=1)
    df_cut_label = df_cut_label.groupby([cut_time_label, cut_lon_label, cut_lat_label]).mean()

    lat_mid = pd.IntervalIndex(df_cut_label.index.get_level_values(2)).mid.unique()
    lon_mid = pd.IntervalIndex(df_cut_label.index.get_level_values(1)).mid.unique()
    time_mid = pd.IntervalIndex(df_cut_label.index.get_level_values(0)).mid.unique()

    df_cut_label.index = df_cut_label.index.set_levels(time_mid.values, level=0)
    df_cut_label.index = df_cut_label.index.set_levels(lon_mid, level=1)
    df_cut_label.index = df_cut_label.index.set_levels(lat_mid.values, level=2)

    df_cut = df_cut_label.copy()
    df_cut.replace(0, np.nan, inplace=True)

    ds_cut = df_cut.to_xarray()
    ds_cut["latitude"] = sorted(lat_mid)
    ds_cut["longitude"] = sorted(lon_mid)
    ds_cut["time"] = time_mid

    return ds_cut, df_cut

ds_merge, df_merge = make_ds_cut(df_merge, nb_bins=39)

### From here we interp ds to have same lat/lon grid as ds_CMA
ds_merge = ds_merge.interp(latitude=ds_CMA.latitude, longitude=ds_CMA.longitude, time=ds_CMA.time, method="nearest")
ds_merge = ds_merge.sel(time=slice(ds_CMA.time.min(), ds_CMA.time.max()))

/tmp/ipykernel_564534/3273259891.py:21: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  bins_dt = pd.date_range(
/tmp/ipykernel_564534/3273259891.py:32: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_cut_label = df_cut_label.groupby([cut_time_label, cut_lon_label, cut_lat_label]).mean()


In [8]:
ds_merge.to_netcdf("/home/jupyter-vincent2/vincent/process_profiles/data/processed_2026/CORA_gridded.nc")

In [9]:
ds = xr.open_dataset("/home/jupyter-vincent2/vincent/process_profiles/data/processed_2026/CORA_gridded.nc")
ds = ds.where(ds.time.dt.year < 2024,drop=True)
df = ds.to_dataframe().reset_index()
### We modify the dataset for the R analysis 
df['year'] = df.time.dt.year
df["mth"] = df.time.dt.month
df = df.drop(columns="time")
df = df.rename(columns={"latitude" : "lat","longitude" : "long"})
df = df.fillna("NA")
df = df[["long","lat","mth","year","mld"]]
df = df.sort_values(["year","mth","long","lat"])
df = df.reset_index(drop=True)

In [10]:
### Number of bins for each month = nb_lat_one_month * nb_long_one_month 
nb_lat_one_month=len(np.unique(df.lat))
nb_long_one_month=len(np.unique(df.long))
nb_bins_one_month = nb_lat_one_month * nb_long_one_month
switch = nb_long_one_month*nb_lat_one_month
switch
df_append = df[:switch].copy()
new_df = df.copy()
new_df

### Before this cell look at which month are missing and for which year
# year = 2024
# months = [2,3,4,5,6,7,8,9,10,11,12]
# for i in months: #months that are missing in our df (need to fill them for the R analysis)
#     df_append = df[:switch].copy()
#     df_append["mth"] = i
#     df_append["year"] = year
#     df_append = df_append.reset_index(drop=True).set_index(np.arange(df.index.max()+1,df.index.max()+(switch+1),1))
#     new_df = pd.concat((new_df,df_append))
#     new_df = new_df.sort_values(["year","mth","long","lat"])
#     new_df =new_df.reset_index(drop=True)

new_df

,long,lat,mth,year,mld
0,60.2465,-59.7525,1,2007,NA
1,60.2465,-59.2300,1,2007,NA
2,60.2465,-58.7175,1,2007,NA
3,60.2465,-58.2045,1,2007,NA
4,60.2465,-57.6915,1,2007,NA
...,...,...,...,...,...
310279,79.7435,-42.3080,12,2023,NA
310280,79.7435,-41.7955,12,2023,NA
310281,79.7435,-41.2825,12,2023,NA
310282,79.7435,-40.7695,12,2023,NA


In [11]:
new_df.to_csv("/home/jupyter-vincent2/vincent/process_profiles/data/R_analysis_2026/raw/CORA.txt",sep=" ",index=False)